# Notebook 3: Ollama + OpenAI Agents SDK — The Best Way

**What you'll learn:** 3 methods to connect Ollama with the Agents SDK, which models work, gotchas, and production patterns for local AI agents.

---

## Why Run Agents Locally?

| Concern | Cloud (OpenAI API) | Local (Ollama) |
|---|---|---|
| Cost | $0.01-0.15/1K tokens | **Free forever** |
| Privacy | Data sent to OpenAI | **Data stays on your machine** |
| Speed | Network latency | **Local inference** |
| Availability | Depends on API uptime | **Always available** |
| Model choice | OpenAI models only | **Any open-source model** |

**The trick:** Ollama exposes an OpenAI-compatible API at `http://localhost:11434/v1`, so the Agents SDK can use it directly!

---

## Prerequisites

### 1. Install Ollama
```bash
# macOS/Linux
curl -fsSL https://ollama.com/install.sh | sh

# Windows — download from https://ollama.com
```

### 2. Pull a model WITH tool-calling support
```bash
# These support tool calling (REQUIRED for agents):
ollama pull qwen2.5:7b        # Great balance of speed + quality
ollama pull llama3.1:8b        # Meta's latest, good tool calling
ollama pull mistral:7b         # Fast and reliable
ollama pull qwen3:8b           # Newest Qwen, excellent tool support

# These do NOT support tool calling (will give 400 errors):
# deepseek-r1         — reasoning model, no tool calling
# phi-2               — too small for reliable tool use
```

### 3. Verify Ollama is running
```bash
ollama list   # Should show your pulled models
curl http://localhost:11434/v1/models  # Should return JSON
```

In [ ]:
# uv add openai-agents openai ollama

---

## Method 1: Direct Model Override (Simplest & Best)

This is the **recommended** approach. Minimal code, maximum clarity.

In [ ]:
# import asyncio
from openai import AsyncOpenAI
from agents import Agent, Runner, OpenAIChatCompletionsModel
from agents.tracing import set_tracing_disabled

# Step 1: Disable tracing (not needed for local models)
set_tracing_disabled(True)

# Step 2: Create an OpenAI-compatible client pointing to Ollama
ollama_client = AsyncOpenAI(
    base_url="http://localhost:11434/v1",   # Ollama's OpenAI-compatible endpoint
    api_key="rizwan"                        # Dummy key — Ollama doesn't need one
)

# Step 3: Wrap it in the SDK's model class
local_model = OpenAIChatCompletionsModel(
    model="llama3.1:8b",         # Must match what you pulled with `ollama pull`
    openai_client=ollama_client
)

# Step 4: Create agent with local model
agent = Agent(
    name="Local Assistant",
    instructions="You are a helpful assistant running locally via Ollama. Be concise.",
    model=local_model             # <-- This is the key line!
)

# Step 5: Run it
result = await Runner.run(agent, "What is the Agents SDK in one paragraph?")
print(result.final_output)

### Key Points:
- `set_tracing_disabled(True)` — Tracing sends data to OpenAI, not needed locally
- `api_key="ollama"` — Any string works, but the client requires something
- The model name must **exactly match** what `ollama list` shows

---

## Method 2: Custom ModelProvider (Reusable Pattern)

Better for larger projects where multiple agents share the same local model setup.

In [ ]:
from openai import AsyncOpenAI
from agents import (
    Agent, Model, ModelProvider,
    OpenAIChatCompletionsModel, Runner,
    set_tracing_disabled
)

set_tracing_disabled(True)

OLLAMA_BASE_URL = "http://localhost:11434/v1"
OLLAMA_MODEL = "llama3.1:8b"


class OllamaProvider(ModelProvider):
    """Custom ModelProvider that routes all model requests to Ollama."""
    
    def __init__(self, model_name: str = OLLAMA_MODEL):
        self.model_name = model_name
        self.client = AsyncOpenAI(
            base_url=OLLAMA_BASE_URL,
            api_key="ollama"
        )
    
    def get_model(self, model_name: str | None = None) -> Model:
        return OpenAIChatCompletionsModel(
            model=model_name or self.model_name,
            openai_client=self.client
        )


# Now create agents WITHOUT specifying model each time
ollama = OllamaProvider()

agent = Agent(
    name="Local Agent",
    instructions="You are a local AI assistant. Be helpful and concise.",
    model=ollama.get_model()  # Clean and reusable!
)

result = await Runner.run(agent, "Explain what Ollama is in 2 sentences.")
print(result.final_output)

### When to use ModelProvider:
- You have 3+ agents using the same local model
- You want to easily switch between Ollama models
- You're building a module/package others will import

---

## Method 3: LiteLLM Integration (100+ Providers)

LiteLLM is a universal adapter. It supports Ollama, Groq, Together, Anthropic, and 100+ more.

In [ ]:
# !pip install litellm

from agents import Agent, Runner, set_tracing_disabled
from agents.extensions.models.litellm_model import LitellmModel

set_tracing_disabled(True)

agent = Agent(
    name="LiteLLM Agent",
    instructions="You are a helpful assistant.",
    # IMPORTANT: prepend 'ollama_chat/' — tells LiteLLM to use Ollama's chat API
    model=LitellmModel(model="ollama_chat/llama3.1:8b", base_url="http://localhost:11434")
)

result = await Runner.run(agent, "Hello! What model are you?")
print(result.final_output)

### Common Gotcha with LiteLLM:
```python
# WRONG — uses completion API (not chat)
model=LitellmModel(model="ollama/qwen2.5:7b")

# CORRECT — uses chat API
model=LitellmModel(model="ollama_chat/qwen2.5:7b")
```

---

## Which Method to Use?

| Method | Best For | Complexity |
|---|---|---|
| **Method 1: Direct** | Learning, quick scripts, single model | 1 Simplest |
| **Method 2: ModelProvider** | Multi-agent projects, shared config | 2 Medium |
| **Method 3: LiteLLM** | Multi-provider (Ollama + Groq + OpenAI) | 3 Most flexible |

---

## Helper: Reusable Ollama Setup Module

Save this as `ollama_setup.py` and import it in all your projects:

In [ ]:
# === ollama_setup.py — Copy this to your projects ===

from openai import AsyncOpenAI
from agents import OpenAIChatCompletionsModel, set_tracing_disabled

# Disable tracing for local models
set_tracing_disabled(True)


def get_ollama_model(
    model_name: str = "llama3.1:8b",
    base_url: str = "http://localhost:11434/v1"
):
    """Create an Ollama-backed model for the Agents SDK.
    
    Args:
        model_name: Ollama model name (must be pulled first)
        base_url: Ollama server URL
    
    Returns:
        OpenAIChatCompletionsModel ready to use with Agent(model=...)
    """
    client = AsyncOpenAI(base_url=base_url, api_key="ollama")
    return OpenAIChatCompletionsModel(model=model_name, openai_client=client)


# Usage:
# from ollama_setup import get_ollama_model
# agent = Agent(name="My Agent", instructions="...", model=get_ollama_model())
# agent = Agent(name="My Agent", instructions="...", model=get_ollama_model("llama3.1:8b"))

print("ollama_setup module ready")

---

## Real-World Pattern: Local Resume Analyzer Agent

**Scenario:** HR department needs to analyze resumes. Data is sensitive — can't send to cloud APIs.

In [ ]:
from pydantic import BaseModel, Field
from typing import Literal
from openai import AsyncOpenAI
from agents import Agent, Runner, OpenAIChatCompletionsModel, set_tracing_disabled


set_tracing_disabled(True)

# Local model — sensitive HR data NEVER leaves the machine
local_model = OpenAIChatCompletionsModel(
    model="llama3.1:8b",
    openai_client=AsyncOpenAI(base_url="http://localhost:11434/v1", api_key="ollama")
)


class ResumeAnalysis(BaseModel):
    name: str = Field(description="Candidate name")
    years_experience: int = Field(description="Total years of experience")
    top_skills: list[str] = Field(description="Top 3-5 relevant skills")
    fit_score: int = Field(description="Job fit score 1-100")
    fit_level: Literal["strong", "moderate", "weak"] = Field(description="Overall fit")
    summary: str = Field(description="2-sentence summary for hiring manager")


resume_agent = Agent(
    name="Resume Analyzer",
    instructions="""
    You are an HR AI assistant that analyzes resumes against job requirements.
    
    JOB: Senior Python Developer
    REQUIREMENTS: 5+ years Python, FastAPI/Django, SQL, cloud experience (AWS/GCP)
    NICE TO HAVE: AI/ML, Docker, Kubernetes, system design
    
    Analyze the resume text and score the candidate's fit honestly.
    """,
    model=local_model,
    output_type=ResumeAnalysis
)

# Simulate a resume
resume_text = """
Ahmed Hassan — Lahore, Pakistan
7 years experience in software development.
Skills: Python, FastAPI, PostgreSQL, Redis, Docker, AWS (EC2, Lambda, S3)
Previous: Lead Developer at SAI Lab (3 years), Senior Dev at SAI (4 years)
Projects: Built a real-time analytics pipeline processing 1M events/day.
Education: BS Computer Science, UAF
"""

result = await Runner.run(resume_agent, resume_text)
r = result.final_output

print(f"{r.name}")
print(f"Experience: {r.years_experience} years")
print(f"Skills: {', '.join(r.top_skills)}")
print(f"Fit Score: {r.fit_score}/100 ({r.fit_level})")
print(f"Summary: {r.summary}")

---

## Troubleshooting Guide

| Error | Cause | Fix |
|---|---|---|
| `Connection refused` | Ollama not running | Run `ollama serve` |
| `400 Bad Request` | Model doesn't support tools | Use qwen2.5, llama3.1, or mistral |
| `Model not found` | Model not pulled | Run `ollama pull model_name` |
| `OPENAI_API_KEY` error | SDK defaults to OpenAI for tracing | Set `set_tracing_disabled(True)` |
| Slow responses | Model too large for hardware | Use smaller model (7b vs 70b) |
| `run_sync` doesn't work | Jupyter has event loop | Use `await Runner.run()` instead |

---

## Summary

| What | How |
|---|---|
| Disable tracing | `set_tracing_disabled(True)` |
| Create Ollama client | `AsyncOpenAI(base_url="http://localhost:11434/v1", api_key="ollama")` |
| Wrap as SDK model | `OpenAIChatCompletionsModel(model="name", openai_client=client)` |
| Use in agent | `Agent(model=local_model)` |
| Tool-capable models | qwen2.5, llama3.1, mistral, qwen3 |

**Next:** Notebook 4 — Mastering Tools (function_tool, Pydantic tools, and real agentic tool patterns).